In [1]:
# !pip uninstall -y transformers datasets tokenizers peft accelerate sacrebleu bitsandbytes -q
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
# !pip install transformers==4.44.2 datasets==2.21.0 peft==0.12.0 sacrebleu==1.5.5 accelerate==0.33.0 tokenizers --no-build-isolation --no-cache-dir -q
# !pip install bitsandbytes nltk --no-build-isolation -q

import pandas as pd, torch, unicodedata, numpy as np, warnings
warnings.filterwarnings("ignore")
from datasets import Dataset
from sklearn.model_selection import train_test_split
import nltk; nltk.download('punkt', quiet=True)
print("✅ Production pipeline ready | GPU:", torch.cuda.is_available())

✅ Production pipeline ready | GPU: True


In [2]:
# from google.colab import files
# uploaded = files.upload()

In [3]:
df = pd.read_csv('sentence_pairs_7870.csv')
df.drop(columns=['Unnamed: 0', 'text_uuid'], inplace=True)
df

,sentence_uuid,transliteration,translation
0,d9c82957-992c-4270-a00e-7cea9902db10,umma Al-aḫum-ma Aššur-idī-ma ana Aššur-nādā u ...,From Alāhum and Aššur-idī to Aššur-nādā and Šū...
1,c0f7ea59-926c-4fd4-8703-9b5f4db8fcd6,ana Aššur-nādā qibima,(specifically) to Aššur-nādā:
2,e3c8821e-cae1-4709-8714-cae2c702a847,ina ūmem ša ṭuppam tašammeāni 1 bilat AN.NA ku...,the same day that you hear the letter Šū-Aššur...
3,b36ff976-b503-4f6c-82e6-ee8e2dead9b8,miššu ša umma attāma,Why is it that you say:
4,b6fbdacb-fada-4e1b-aec0-d46706c6815f,taṣbat,"""You have seized my silver""?"
...,...,...,...
7865,243a3e54-87df-44e2-a79b-a3f337b06c9e,maḫar Aššur-lamassī maḫar Uzuwa maḫar Inar maḫ...,"Witnesses: Aššur-lamassī, Uzuwa, Inar, Ḫappi-aḫšu"
7866,689e96ec-ab48-4c45-9faf-2eb33b228d42,iṣṣēr Kunānīya Lāqēp īšu,Kunaniya owes Lā-qēpum 1 mina 1 1/2 shekels si...
7867,65f83adc-2669-429b-b646-e6d556973e9b,ištu ḫamuštem ša Ikūnim u Puzur-Aššur ana 5 wa...,He will pay 5 months from the ḫamuštum week of...
7868,ab0c1856-7f8d-4fae-96e2-fd8606566771,uṣṣab,If he does not pay at the fulfillment of the t...


In [4]:
# df = pd.read_csv('sentence_pairs_7870.csv')  # Upload your file first
# print("Original columns:", df.columns.tolist())

# Rename to standard (adjust based on your columns)
column_map = {
    'transliteration': 'text',  # Akkadian
    'translation': 'label'     # English
}
df = df.rename(columns={k: v for k, v in column_map.items() if k in df.columns})
df = df[['text', 'label']].dropna().drop_duplicates()
df['text'] = df['text'].apply(lambda x: unicodedata.normalize('NFKC', str(x)))

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# **from_dict() - NO index column EVER**
train_dataset = Dataset.from_dict({
    'text': train_df['text'].tolist(),
    'label': train_df['label'].tolist()
})
val_dataset = Dataset.from_dict({
    'text': val_df['text'].tolist(),
    'label': val_df['label'].tolist()
})

print(f"✅ Clean: Train={len(train_dataset):,} Val={len(val_dataset):,} | Sample: {train_dataset[0]['text'][:50]}")

✅ Clean: Train=6,857 Val=762 | Sample: iṣṣēr Iddin-abim Amur-Ištar i-šu-ú


In [6]:
uploaded = files.upload()
uploaded = files.upload()

Saving final_processed_dict_df_satvik.csv to final_processed_dict_df_satvik.csv


Saving pos_train.csv to pos_train.csv


In [5]:
dict_df = pd.read_csv('final_processed_dict_df_satvik.csv')  # Columns: 'akkadian_word', 'english_meaning'
pos_df = pd.read_csv('pos_train.csv')
print(dict_df.columns)
print(pos_df.columns)

Index(['Unnamed: 0', 'word', 'definition', 'derived_from', 'Meanings',
       'Quality', 'Source'],
      dtype='object')
Index(['Unnamed: 0', 'oare_id', 'transliteration', 'translation',
       'normalized_text', 'norm_tokens', 'pos_tags'],
      dtype='object')


In [6]:
import pandas as pd
import ast
import re

# Load your CSV files (upload via Files tab)
dict_df = pd.read_csv('final_processed_dict_df_satvik.csv')  # word, meanings, quality
pos_df = pd.read_csv('pos_train.csv')     # transliteration, pos_tags

print("📊 Raw data:")
print(f"Dictionary: {len(dict_df)} entries")
print(f"POS tags: {len(pos_df)} sentences")
print("Sample POS:", pos_df['pos_tags'].iloc[0][:200] + "...")

# Process dictionary (quality-filtered)
def parse_meanings(meanings_str):
    try:
        meanings_str = meanings_str.strip('"\'')
        if meanings_str.startswith('['):
            return ast.literal_eval(meanings_str)
        return [meanings_str]
    except:
        return [meanings_str]

dict_df['Meanings'] = dict_df['Meanings'].apply(parse_meanings)
high_quality_dict = {row['word']: row['Meanings'][0] for _, row in dict_df[dict_df['Quality']=='High'].iterrows()}
medium_quality_dict = {row['word']: row['Meanings'][0] for _, row in dict_df[dict_df['Quality']=='Medium'].iterrows()}

# Process POS tuples: {'transliteration': [(word1, tag1), (word2, tag2), ...]}
pos_lookup = {}
for _, row in pos_df.iterrows():
    try:
        # Parse tuple list from string
        pos_tuples = ast.literal_eval(row['pos_tags'])
        # Create lookup: word → tag
        word_tags = {word: tag for word, tag in pos_tuples}
        pos_lookup[row['transliteration']] = word_tags
    except:
        continue

print(f"✅ Processed:")
print(f"   High-quality dict: {len(high_quality_dict)} words")
print(f"   POS lookup: {len(pos_lookup)} sentences")
print(f"   Sample: {list(high_quality_dict.items())[:3]}")


📊 Raw data:
Dictionary: 15339 entries
POS tags: 1561 sentences
Sample POS: [('kunukkum', 'word'), ('Mannum-bālum-Aššur', 'PN'), ("mer'ēn", 'word'), ('ṣí-lá-dIM', 'UNK'), ('kunukkum', 'word'), ('šu-dENLÍL', 'UNK'), ("mer'ēn", 'word'), ('Mannum-kī-Aššur', 'PN'), ('kunukkum', '...
✅ Processed:
   High-quality dict: 10817 words
   POS lookup: 1559 sentences
   Sample: [('-a', 'my'), ('-am', 'to me'), ('-iš', 'to; like')]


In [7]:
from transformers import AutoTokenizer
model_name = "google/byt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token

def smart_multi_augment(examples):
    """4x data: direct + dict + POS + dict+POS"""
    all_inputs, all_labels = [], []

    for text, label in zip(examples['text'], examples['label']):
        words = text.split()

        # 1. DIRECT TRANSLATION (40%)
        all_inputs.append(f"translate Akkadian to English: {text}")
        all_labels.append(label)

        # 2. HIGH-QUALITY DICTIONARY (20%)
        dict_aug = dictionary_replace(words, high_quality_dict)
        if dict_aug != ' '.join(words):
            all_inputs.append(f"gloss-enhanced: {dict_aug}")
            all_labels.append(label)

        # 3. POS TUPLE ANNOTATION (20%)
        pos_aug = add_pos_annotations(text, pos_lookup)
        all_inputs.append(f"POS-tagged: {pos_aug}")
        all_labels.append(label)

        # 4. DICT + POS COMBO (20%)
        dict_pos = dictionary_replace(words, high_quality_dict)
        dict_pos = add_pos_annotations(dict_pos, pos_lookup)
        all_inputs.append(f"full-augmented: {dict_pos}")
        all_labels.append(label)

    # Tokenize
    model_inputs = tokenizer(all_inputs, max_length=512, truncation=True)
    label_inputs = tokenizer(all_labels, max_length=512, truncation=True)
    model_inputs['labels'] = label_inputs['input_ids']
    return model_inputs

def dictionary_replace(words, dictionary):
    """Replace Akkadian → English glosses"""
    enhanced = []
    for word in words:
        clean_word = re.sub(r'[^\w]', '', word.lower())
        if clean_word in dictionary:
            enhanced.append(f"{word}[{dictionary[clean_word]}]")
        else:
            enhanced.append(word)
    return ' '.join(enhanced)

def add_pos_annotations(text, pos_lookup):
    """Match sentence → POS tuples → annotate words"""
    clean_text = text.strip().lower()

    # Exact sentence match first
    if clean_text in pos_lookup:
        word_tags = pos_lookup[clean_text]
        words = text.split()
        annotated = []
        for word in words:
            clean_word = re.sub(r'[^\w]', '', word.lower())
            tag = word_tags.get(clean_word, 'UNK')
            annotated.append(f"{word}_{tag}")
        return ' '.join(annotated)

    # Fallback: word-level POS
    words = text.split()
    annotated = []
    for word in words:
        clean_word = re.sub(r'[^\w]', '', word.lower())
        # Simple heuristic if no exact match
        tag = 'word' if clean_word in high_quality_dict else 'UNK'
        annotated.append(f"{word}_{tag}")
    return ' '.join(annotated)

# Generate 4x augmented data!
print("🔄 Creating 4x augmented dataset...")
train_ds = train_dataset.map(smart_multi_augment, batched=True, batch_size=64, remove_columns=['text', 'label'])
val_ds = val_dataset.map(smart_multi_augment, batched=True, batch_size=64, remove_columns=['text', 'label'])

print(f"✅ Augmented dataset ready:")
print(f"   Train: {len(train_ds):,} (4x original)")
print(f"   Val: {len(val_ds):,}")
print(f"   Sample: {tokenizer.decode(train_ds[0]['input_ids'], skip_special_tokens=True)[:200]}...")

🔄 Creating 4x augmented dataset...


Map:   0%|          | 0/6857 [00:00<?, ? examples/s]

Map:   0%|          | 0/762 [00:00<?, ? examples/s]

✅ Augmented dataset ready:
   Train: 25,083 (4x original)
   Val: 2,780
   Sample: translate Akkadian to English: iṣṣēr Iddin-abim Amur-Ištar i-šu-ú...


In [ ]:
# from peft import LoraConfig, get_peft_model, TaskType
# from transformers import Trainer, TrainingArguments

# peft_config = LoraConfig(
#     task_type=TaskType.SEQ_2_SEQ_LM,
#     r=16, lora_alpha=32, lora_dropout=0.1,
#     target_modules=["q", "k", "v", "o"]
# )
# model = get_peft_model(model, peft_config)

# args = TrainingArguments(
#     output_dir='./byt5-akkadian-colab',
#     num_train_epochs=3,
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=4,
#     warmup_steps=100,
#     logging_steps=10,
#     eval_strategy="steps",
#     eval_steps=50,
#     save_strategy="steps",
#     save_steps=100,
#     learning_rate=2e-4,
#     fp16=True,
#     save_total_limit=2,
#     dataloader_num_workers=0,
#     remove_unused_columns=False  # Keep text columns for collator
# )

# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     tokenizer=tokenizer,
#     data_collator=data_collator
# )
# trainer.train()

In [ ]:
# model = AutoModelForSeq2SeqLM.from_pretrained(
#     model_name,
#     load_in_8bit=True,
#     device_map="auto",
#     torch_dtype=torch.float16
# )

# peft_config = LoraConfig(
#     task_type=TaskType.SEQ_2_SEQ_LM,
#     inference_mode=False,
#     r=16,
#     lora_alpha=32,
#     lora_dropout=0.1,
#     target_modules=["q", "k", "v", "o"]
# )
# model = get_peft_model(model, peft_config)
# model.print_trainable_parameters()

# args = TrainingArguments(
#     output_dir='./byt5-akkadian-colab',
#     num_train_epochs=3,
#     per_device_train_batch_size=4,
#     per_device_eval_batch_size=4,
#     gradient_accumulation_steps=4,
#     warmup_steps=100,
#     logging_steps=10,
#     eval_strategy="steps",
#     eval_steps=50,
#     save_steps=100,
#     learning_rate=2e-4,
#     fp16=True,
#     save_total_limit=2,
#     report_to=None,
#     dataloader_num_workers=0  # Colab stability
# )
# from transformers import DataCollatorForSeq2Seq

# data_collator = DataCollatorForSeq2Seq(
#     tokenizer=tokenizer,
#     model=model,
#     padding=True,  # Dynamic padding per batch
#     return_tensors="pt",
#     label_pad_token_id=-100  # Ignore padding in loss
# )

# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     tokenizer=tokenizer
# )
# trainer.train()